In [1]:

# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import librosa
import numpy as np
import pandas as pd
import pyloudnorm as pyln
import warnings

warnings.filterwarnings('ignore')
print("✅ Imports done")


✅ Imports done


In [2]:

# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
BASE_DIR      = "/Users/abey/Documents/amplitude"   # ← change this
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

# delta tolerances
LUFS_TOLERANCE     = 4.0    # LUFS — calibrate with editor later
LRA_TOLERANCE      = 3.0    # LUFS
CENTROID_TOLERANCE = 500    # Hz

# absolute thresholds
PEAK_LIMIT         = -1.0   # dBFS — TTS clipping threshold

# reference quality thresholds — outside this range = degraded
REF_LUFS_MIN       = -40.0
REF_LUFS_MAX       = -5.0

print(f"LUFS tolerance     : ±{LUFS_TOLERANCE} LUFS")
print(f"LRA tolerance      : ±{LRA_TOLERANCE} LUFS")
print(f"Centroid tolerance : ±{CENTROID_TOLERANCE} Hz")
print(f"Peak limit         : {PEAK_LIMIT} dBFS")
print(f"Ref LUFS range     : {REF_LUFS_MIN} to {REF_LUFS_MAX} LUFS")
print("✅ Paths and thresholds set")




LUFS tolerance     : ±4.0 LUFS
LRA tolerance      : ±3.0 LUFS
Centroid tolerance : ±500 Hz
Peak limit         : -1.0 dBFS
Ref LUFS range     : -40.0 to -5.0 LUFS
✅ Paths and thresholds set


In [3]:

# ============================================================
# CELL 3 — analyze_audio function
# ============================================================
def analyze_audio(file_path):
    """
    Returns LUFS, LRA, spectral centroid, true peak.
    librosa loads as (channels, samples) — must transpose to
    (samples, channels) for pyloudnorm.
    """
    data, sr = librosa.load(file_path, sr=None, mono=False)

    # librosa mono returns 1D — pyloudnorm needs 2D (samples, channels)
    if data.ndim == 1:
        data_pln = data.reshape(-1, 1)
    else:
        data_pln = data.T  # (channels, samples) → (samples, channels)

    meter = pyln.Meter(sr)
    lufs  = meter.integrated_loudness(data_pln)
    lra   = meter.loudness_range(data_pln)

    # spectral centroid on mono
    mono     = data if data.ndim == 1 else librosa.to_mono(data)
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=mono, sr=sr)))

    peak_amp = float(np.max(np.abs(data_pln)))
    peak_db  = 20 * np.log10(peak_amp) if peak_amp > 0 else -100.0

    return round(lufs, 3), round(lra, 3), round(centroid, 2), round(peak_db, 3)

print("✅ analyze_audio defined")



✅ analyze_audio defined


In [4]:

# ============================================================
# CELL 4 — Startup validation
# ============================================================
for folder in [BASE_DIR, REFERENCE_DIR, MODELS_DIR]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")
print("✅ Top level folders found")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([f for f in os.listdir(model_path) if f.endswith(".wav")])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

# cross-model filename validation
reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
for wav_file in sample_names:
    ref_path = os.path.join(REFERENCE_DIR, wav_file)
    if not os.path.exists(ref_path):
        raise FileNotFoundError(f"Missing reference for {wav_file} — expected: {ref_path}")
print("✅ All reference files found")

total = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")


✅ Top level folders found
✅ Models found: ['m1']
   m1: 2 samples
✅ All models have identical filenames
✅ All reference files found

Ready: 1 models × 2 samples = 2 evaluations


In [5]:

# ============================================================
# CELL 5 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)
        ref_path    = os.path.join(REFERENCE_DIR, wav_file)

        print(f"\n  Sample : {sample_name}")

        try:
            ref_lufs, ref_lra, ref_cent, ref_peak = analyze_audio(ref_path)
            tts_lufs, tts_lra, tts_cent, tts_peak = analyze_audio(tts_path)

            print(f"  Ref    : LUFS={ref_lufs} | LRA={ref_lra} | Cent={ref_cent} | Peak={ref_peak}")
            print(f"  TTS    : LUFS={tts_lufs} | LRA={tts_lra} | Cent={tts_cent} | Peak={tts_peak}")

            # ── reference quality check — degraded if LUFS out of range ──
            ref_lufs_degraded = not (REF_LUFS_MIN <= ref_lufs <= REF_LUFS_MAX)
            is_degraded       = ref_lufs_degraded
            ref_flag          = "REF_LUFS_DEGRADED" if ref_lufs_degraded else "—"

            # ── true peak — always absolute, always runs ──
            tts_clipping  = tts_peak >= PEAK_LIMIT
            ref_clipping  = ref_peak >= PEAK_LIMIT

            if tts_clipping and ref_clipping:
                peak_flag = "REF_ALSO_CLIPPED"   # informational — TTS still fails
            elif tts_clipping:
                peak_flag = "TTS_CLIPPING"
            elif ref_clipping:
                peak_flag = "REF_CLIPPED"         # informational only
            else:
                peak_flag = "—"

            # ── delta checks — only on clean segments ──
            lufs_diff = round(abs(ref_lufs - tts_lufs), 3)
            lra_diff  = round(abs(ref_lra  - tts_lra),  3)
            cent_diff = round(abs(ref_cent - tts_cent),  2)

            failures = []

            # true peak always counts
            if tts_clipping:
                failures.append("Clipping")

            if not is_degraded:
                if lufs_diff > LUFS_TOLERANCE:
                    failures.append("Volume")
                if lra_diff > LRA_TOLERANCE:
                    failures.append("Dynamics")
                if cent_diff > CENTROID_TOLERANCE:
                    failures.append("EQ")

            final_pass = "✅ PASS" if not failures else f"❌ FAIL ({', '.join(failures)})"

            print(f"  Result : {final_pass} | Degraded: {is_degraded} | Peak: {peak_flag}")

            results.append({
                "Model"        : model,
                "Sample"       : sample_name,
                "Ref LUFS"     : ref_lufs,
                "TTS LUFS"     : tts_lufs,
                "LUFS Delta"   : lufs_diff,
                "Ref LRA"      : ref_lra,
                "TTS LRA"      : tts_lra,
                "LRA Delta"    : lra_diff,
                "Ref Cent"     : ref_cent,
                "TTS Cent"     : tts_cent,
                "Cent Delta"   : cent_diff,
                "Ref Peak"     : ref_peak,
                "TTS Peak"     : tts_peak,
                "Peak Flag"    : peak_flag,
                "Final Pass"   : final_pass,
                "Ref Flag"     : ref_flag,
                "_is_degraded" : is_degraded,
            })

        except Exception as e:
            print(f"  🚨 ERROR: {e}")
            results.append({
                "Model"        : model,
                "Sample"       : sample_name,
                "Ref LUFS"     : None,
                "TTS LUFS"     : None,
                "LUFS Delta"   : None,
                "Ref LRA"      : None,
                "TTS LRA"      : None,
                "LRA Delta"    : None,
                "Ref Cent"     : None,
                "TTS Cent"     : None,
                "Cent Delta"   : None,
                "Ref Peak"     : None,
                "TTS Peak"     : None,
                "Peak Flag"    : "ERROR",
                "Final Pass"   : "⚠️ ERROR",
                "Ref Flag"     : "ERROR",
                "_is_degraded" : False,
            })

print("\n\nAll evaluations complete.")





Model: m1

  Sample : YASH 2
  Ref    : LUFS=-20.003 | LRA=12.377 | Cent=2252.73 | Peak=0.0
  TTS    : LUFS=-17.355 | LRA=16.514 | Cent=2137.82 | Peak=0.0
  Result : ❌ FAIL (Clipping, Dynamics) | Degraded: False | Peak: REF_ALSO_CLIPPED

  Sample : YASH_01
  Ref    : LUFS=-17.355 | LRA=16.514 | Cent=2137.82 | Peak=0.0
  TTS    : LUFS=-20.003 | LRA=12.377 | Cent=2252.73 | Peak=0.0
  Result : ❌ FAIL (Clipping, Dynamics) | Degraded: False | Peak: REF_ALSO_CLIPPED


All evaluations complete.


In [6]:

# ============================================================
# CELL 6 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment results ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample",
    "Ref LUFS", "TTS LUFS", "LUFS Delta",
    "Ref LRA",  "TTS LRA",  "LRA Delta",
    "Ref Cent", "TTS Cent", "Cent Delta",
    "Ref Peak", "TTS Peak", "Peak Flag",
    "Final Pass", "Ref Flag"
]].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df    = df[df["Model"] == model]
    clean_df    = model_df[~model_df["_is_degraded"] & (model_df["Final Pass"] != "⚠️ ERROR")]
    degraded_df = model_df[model_df["_is_degraded"]]
    total       = len(model_df)

    clean_total = len(clean_df)
    clean_pass  = (clean_df["Final Pass"] == "✅ PASS").sum()

    deg_total   = len(degraded_df)
    deg_pass    = (degraded_df["Final Pass"] == "✅ PASS").sum()

    clipping_count = model_df["Final Pass"].str.contains("Clipping").sum()
    volume_count   = model_df["Final Pass"].str.contains("Volume").sum()
    dynamics_count = model_df["Final Pass"].str.contains("Dynamics").sum()
    eq_count       = model_df["Final Pass"].str.contains("EQ").sum()
    error_count    = (model_df["Final Pass"] == "⚠️ ERROR").sum()

    summary_rows.append({
        "Model"             : model,
        "Total Segments"    : total,
        "Clean Segments"    : clean_total,
        "Clean Pass Rate"   : f"{clean_pass}/{clean_total}"  if clean_total > 0 else "—",
        "Degraded Segments" : deg_total,
        "Degraded Pass Rate": f"{deg_pass}/{deg_total}"      if deg_total > 0 else "—",
        "Clipping Fails"    : clipping_count,
        "Volume Fails"      : volume_count,
        "Dynamics Fails"    : dynamics_count,
        "EQ Fails"          : eq_count,
        "Errors"            : error_count,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Clean Pass Rate (ref LUFS within -40 to -5 segments only)")
print("Tiebreak1 → Degraded Pass Rate")
print("Tiebreak2 → Clipping count (lowest first)\n")

def parse_rate(rate_str):
    if rate_str == "—":
        return -1
    return int(rate_str.split("/")[0])

summary_df["_clean_pass_num"]    = summary_df["Clean Pass Rate"].apply(parse_rate)
summary_df["_degraded_pass_num"] = summary_df["Degraded Pass Rate"].apply(parse_rate)

ranking = summary_df.sort_values(
    by=["_clean_pass_num", "_degraded_pass_num", "Clipping Fails"],
    ascending=[False, False, True]
)[[
    "Model", "Clean Pass Rate", "Degraded Pass Rate",
    "Clipping Fails", "Volume Fails", "Dynamics Fails", "EQ Fails"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Clean Pass Rate    → primary ranking — ref LUFS within -40 to -5 only")
print("Degraded Pass Rate → segments where reference loudness was abnormal")
print("Clipping Fails     → TTS peak >= -1.0 dBFS — always fails regardless of ref")
print("REF_ALSO_CLIPPED   → both clipped — TTS still fails, ref state is informational")
print("REF_CLIPPED        → ref clipped but TTS clean — informational only")
print("Volume Fails       → LUFS delta > 4.0 — loudness mismatch")
print("Dynamics Fails     → LRA delta > 3.0 — dynamic range mismatch")
print("EQ Fails           → centroid delta > 500Hz — tonal brightness mismatch")
print(f"\nThresholds:")
print(f"  LUFS tolerance     : ±{LUFS_TOLERANCE} LUFS")
print(f"  LRA tolerance      : ±{LRA_TOLERANCE} LUFS")
print(f"  Centroid tolerance : ±{CENTROID_TOLERANCE} Hz")
print(f"  Peak limit         : {PEAK_LIMIT} dBFS")
print(f"  Ref LUFS range     : {REF_LUFS_MIN} to {REF_LUFS_MAX} LUFS")



========== FULL PER-SEGMENT RESULTS ==========
Model  Sample  Ref LUFS  TTS LUFS  LUFS Delta  Ref LRA  TTS LRA  LRA Delta  Ref Cent  TTS Cent  Cent Delta  Ref Peak  TTS Peak        Peak Flag                  Final Pass Ref Flag
   m1  YASH 2   -20.003   -17.355       2.648   12.377   16.514      4.137   2252.73   2137.82      114.91       0.0       0.0 REF_ALSO_CLIPPED ❌ FAIL (Clipping, Dynamics)        —
   m1 YASH_01   -17.355   -20.003       2.648   16.514   12.377      4.137   2137.82   2252.73      114.91       0.0       0.0 REF_ALSO_CLIPPED ❌ FAIL (Clipping, Dynamics)        —

========== MODEL COMPARISON SUMMARY ==========
Model  Total Segments  Clean Segments Clean Pass Rate  Degraded Segments Degraded Pass Rate  Clipping Fails  Volume Fails  Dynamics Fails  EQ Fails  Errors
   m1               2               2             0/2                  0                  —               2             0               2         0       0

========== MODEL RANKING ==========
Primary   → 

In [7]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
